# 02 · Tratamiento de datos financieros con Python
### Consulta, exploración y preparación de precios históricos

**Financial Analytics Lab · Python · Google Colab**

**Pregunta orientadora:** ¿Qué debe verificarse antes de utilizar los precios históricos de un activo para realizar cálculos financieros?

En el módulo anterior calculamos rendimientos con precios hipotéticos. Ahora consultaremos información histórica del mercado, reconoceremos su estructura y construiremos una base de datos que pueda analizarse y reproducirse.

**Aprenderás a:** identificar una fuente de información; utilizar `yfinance`; trabajar con `pandas` y DataFrames; reconocer fechas, índices y tipos de datos; examinar datos faltantes y duplicados; seleccionar y transformar columnas; y documentar decisiones de limpieza.

**Cómo trabajar:** ejecuta las celdas en orden. Los bloques «Tu práctica» deben completarse con tus propias decisiones y explicaciones. Este notebook requiere conexión a internet para descargar datos; la disponibilidad y formato de los resultados pueden cambiar.

**Nota metodológica:** los precios históricos describen el pasado y no constituyen recomendaciones de inversión.

## 0 · El problema de información

Un analista desea estudiar el comportamiento de las acciones de Microsoft (`MSFT`) durante un período determinado. Antes de calcular rendimientos debe responder: ¿de dónde provienen los datos?, ¿qué representa cada fila?, ¿qué significan las columnas?, ¿existen observaciones faltantes? y ¿son comparables las fechas?

| Etapa | Pregunta | Resultado esperado |
|---|---|---|
| Extracción | ¿Cómo obtener los datos? | Serie histórica descargada |
| Exploración | ¿Qué contiene la tabla? | Descripción de filas, columnas y tipos |
| Validación | ¿Qué problemas presenta? | Diagnóstico de calidad |
| Transformación | ¿Qué datos necesitamos? | Serie preparada para análisis |
| Documentación | ¿Qué decisiones tomamos? | Procedimiento reproducible |

Este proceso forma parte del enfoque **ETL (extracción, transformación y carga)**. En este cuaderno prepararemos un conjunto de datos para los módulos siguientes.

## 1 · Herramientas: pandas y yfinance

**pandas** es una biblioteca de Python para trabajar con información estructurada. Su objeto `DataFrame` organiza datos en filas y columnas; una `Series` representa una secuencia unidimensional con etiquetas.

**yfinance** permite consultar información financiera disponible a través de Yahoo Finance. No es una fuente oficial de cotizaciones ni garantiza continuidad o disponibilidad del servicio. Para un trabajo reproducible debemos registrar el activo, las fechas, el momento de consulta y las opciones de descarga.

**Documentación:** [pandas](https://pandas.pydata.org/docs/) · [yfinance](https://ranaroussi.github.io/yfinance/) · [Google Colab](https://colab.research.google.com/).

In [ ]:
%pip -q install yfinance pandas matplotlib

In [ ]:
import pandas as pd
import yfinance as yf
from datetime import datetime, timezone

print("pandas:", pd.__version__)
print("yfinance:", yf.__version__)
print("Consulta UTC:", datetime.now(timezone.utc).isoformat(timespec="seconds"))

## 2 · Consultar una serie histórica

Un **ticker** identifica un instrumento financiero en una fuente de información. Utilizaremos `MSFT` como ejemplo y fechas explícitas para facilitar la repetición del análisis.

En `yf.download()`, `start` define la fecha inicial incluida y `end` establece el límite final **exclusivo**. `auto_adjust=True` solicita precios ajustados por los eventos corporativos que contempla la fuente; por ello no debe interpretarse la columna `Close` sin registrar esta configuración.

**Diferencia importante:** `period="1y"` utiliza un intervalo relativo al momento de la consulta; `start` y `end` delimitan fechas específicas. Evita combinarlos en una misma consulta.

In [ ]:
ticker = "MSFT"
fecha_inicio = "2024-01-01"
fecha_fin = "2025-01-01"  # Límite exclusivo

datos = yf.download(
    tickers=ticker,
    start=fecha_inicio,
    end=fecha_fin,
    auto_adjust=True,
    progress=False,
    multi_level_index=False
)

if datos.empty:
    raise RuntimeError("La consulta no devolvió datos. Verifica internet, ticker, fechas y disponibilidad del servicio.")

print("Dimensiones (filas, columnas):", datos.shape)
display(datos.head())

**Interpretación:** cada fila corresponde a una observación fechada y las columnas representan variables del mercado. `Open`, `High`, `Low`, `Close` y `Volume` pueden estar disponibles, pero su presencia debe comprobarse en cada descarga.

**Tu práctica 1:** modifica el ticker por `AAPL` y consulta el mismo período. Registra el número de observaciones y explica por qué puede diferir del número de días calendario.

In [ ]:
# TU PRÁCTICA 1
# Descarga otro activo y documenta ticker, fechas, cantidad de filas y columnas.

## 3 · Comprender un DataFrame

Un DataFrame tiene **índice**, **columnas** y **valores**. En esta consulta, el índice contiene fechas; `head()` muestra los primeros registros, `tail()` los últimos, `dtypes` informa los tipos de datos e `info()` resume la estructura.

La fecha es parte del significado financiero: no debemos tratarla como una simple etiqueta decorativa.

In [ ]:
print("Índice:", type(datos.index).__name__)
print("Primer registro:", datos.index.min())
print("Último registro:", datos.index.max())
print("Columnas:", list(datos.columns))
print("\nTipos de datos:")
print(datos.dtypes)
print("\nÚltimas observaciones:")
display(datos.tail())

### Seleccionar observaciones

`loc` permite seleccionar por etiquetas o fechas; `iloc` selecciona por posición. Las fechas deben estar ordenadas cronológicamente antes de efectuar operaciones temporales.

In [ ]:
datos = datos.sort_index()
display(datos.loc["2024-03-01":"2024-03-15"].head())
display(datos.iloc[:3])

**Tu práctica 2:** explica la diferencia entre `loc` e `iloc`, identifica la variable que usarías como precio de cierre y muestra las últimas cinco observaciones.

In [ ]:
# TU PRÁCTICA 2
# Utiliza loc, iloc y tail; documenta la interpretación.

## 4 · Validar la calidad de los datos

Una tabla descargada no está automáticamente lista para analizarse. Debemos comprobar que las fechas no estén repetidas, que los valores esenciales estén disponibles y que las columnas tengan tipos adecuados.

Un dato faltante (`NaN`) no equivale a un precio igual a cero. Tampoco debe rellenarse automáticamente con cero: esa decisión introduciría información que no observamos.

In [ ]:
print("Fechas duplicadas:", datos.index.duplicated().sum())
print("Valores faltantes por columna:")
display(datos.isna().sum().to_frame("faltantes"))
print("Índice ordenado:", datos.index.is_monotonic_increasing)
print("Tipo del índice:", type(datos.index).__name__)

if "Close" not in datos.columns:
    raise KeyError("No existe la columna Close. Revisa la estructura devuelta por la fuente.")
print("Precios de cierre no positivos:", (datos["Close"].dropna() <= 0).sum())

### ¿Cómo decidir una transformación?

No todas las ausencias requieren la misma solución. Si falta un precio esencial, podemos excluir esa observación **después de cuantificar el problema y documentar la decisión**. Si faltan días no bursátiles, eso no significa que debamos crear precios ficticios para sábados o domingos.

Para nuestro primer análisis utilizaremos las observaciones con un cierre válido y positivo. Conservaremos una copia de los datos originales para no perder trazabilidad.

In [ ]:
datos_originales = datos.copy()
serie = datos[["Close"]].copy()
filas_antes = len(serie)
serie = serie.dropna(subset=["Close"])
serie = serie.loc[serie["Close"] > 0]
serie = serie.loc[~serie.index.duplicated(keep="first")]
serie = serie.sort_index()
print("Filas originales:", filas_antes)
print("Filas válidas:", len(serie))
print("Filas excluidas:", filas_antes - len(serie))
display(serie.head())

**Tu práctica 3:** registra cuántas filas se excluyeron y justifica el criterio. Si no se excluyó ninguna, explica por qué sigue siendo importante ejecutar las validaciones.

In [ ]:
# TU PRÁCTICA 3
# Escribe aquí tu diagnóstico de calidad y las decisiones de preparación.

## 5 · Transformar: calcular rendimientos simples

Una vez ordenados y validados los precios, podemos calcular su variación relativa entre observaciones consecutivas:

**Rendimiento simple:** \(r_t=(P_t/P_{t-1})-1\).

`pct_change()` calcula esta variación relativa. Multiplicar por 100 la expresa en porcentaje. El primer rendimiento queda sin valor porque no hay un precio anterior dentro de la muestra.

**Precaución:** el rendimiento corresponde al intervalo entre dos observaciones disponibles; si existen días sin negociación, no debe interpretarse automáticamente como un rendimiento de un día calendario.

In [ ]:
serie["Rendimiento"] = serie["Close"].pct_change(fill_method=None)
serie["Rendimiento_pct"] = serie["Rendimiento"] * 100
display(serie.head(8))
print("Rendimientos válidos:", serie["Rendimiento"].notna().sum())

### Verificar el cálculo

El resultado de una función debe poder contrastarse con su definición. Calcularemos manualmente el rendimiento entre los dos primeros cierres y lo compararemos con pandas.

In [ ]:
if len(serie) < 2:
    raise RuntimeError("Se necesitan al menos dos precios para verificar un rendimiento.")
precio_0 = serie["Close"].iloc[0]
precio_1 = serie["Close"].iloc[1]
rendimiento_manual = precio_1 / precio_0 - 1
rendimiento_pandas = serie["Rendimiento"].iloc[1]
print(f"Manual: {rendimiento_manual:.8f}")
print(f"pandas: {rendimiento_pandas:.8f}")
assert abs(rendimiento_manual - rendimiento_pandas) < 1e-10
print("Verificación superada.")

**Interpretación:** un rendimiento positivo indica que el cierre aumentó respecto de la observación anterior; uno negativo indica una disminución. No debe confundirse el **precio** (USD por acción) con el **rendimiento** (proporción o porcentaje).

**Tu práctica 4:** selecciona otras dos observaciones consecutivas, calcula el rendimiento manualmente y contrástalo con `pct_change()`. Explica el signo del resultado.

In [ ]:
# TU PRÁCTICA 4
# Selecciona dos cierres consecutivos y verifica su rendimiento.

## 6 · Primer gráfico de precios y rendimientos

Una visualización debe responder una pregunta. El gráfico de precios permite examinar la evolución del nivel de cotización; el de rendimientos permite identificar cambios relativos y episodios de mayor variación. No son representaciones intercambiables.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(serie.index, serie["Close"], linewidth=1.5)
ax.set(title=f"Precio de cierre ajustado · {ticker}", xlabel="Fecha", ylabel="USD por acción")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(serie.index, serie["Rendimiento_pct"], linewidth=0.9)
ax.axhline(0, linewidth=1)
ax.set(title=f"Rendimiento entre observaciones · {ticker}", xlabel="Fecha", ylabel="Rendimiento (%)")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

**Tu práctica 5:** redacta una interpretación de cada gráfico. Indica qué se puede observar y qué no puede concluirse únicamente a partir de estas representaciones.

In [ ]:
# TU PRÁCTICA 5
# Puedes crear un gráfico adicional con un período más corto.

## 7 · Preparar un conjunto de datos de varios activos

En un portafolio debemos comparar observaciones de distintos instrumentos. La primera dificultad es **alinear las fechas**: si dos activos tienen registros en fechas diferentes, no debemos tratarlos como si fueran simultáneos.

Descargaremos tres activos y construiremos una tabla de cierres. `join="inner"` conserva únicamente las fechas presentes en todas las series válidas. Esta decisión facilita comparaciones, pero reduce el número de observaciones; por eso debe documentarse.

In [ ]:
tickers = ["MSFT", "AAPL", "JPM"]
series_activos = []
diagnostico = []

for simbolo in tickers:
    tabla = yf.download(
        simbolo, start=fecha_inicio, end=fecha_fin,
        auto_adjust=True, progress=False, multi_level_index=False
    )
    if tabla.empty or "Close" not in tabla.columns:
        raise RuntimeError(f"No se pudo consultar el cierre de {simbolo}.")
    cierre = tabla[["Close"]].copy().sort_index()
    cierre = cierre.loc[~cierre.index.duplicated(keep="first")]
    cierre = cierre.dropna(subset=["Close"])
    cierre = cierre.loc[cierre["Close"] > 0]
    diagnostico.append({"Ticker": simbolo, "Observaciones válidas": len(cierre)})
    series_activos.append(cierre.rename(columns={"Close": simbolo}))

precios = pd.concat(series_activos, axis=1, join="inner").sort_index()
precios = precios.dropna()
if precios.empty:
    raise RuntimeError("No hay fechas comunes para comparar los activos.")
print("Diagnóstico por activo:")
display(pd.DataFrame(diagnostico))
print("Fechas comunes:", len(precios))
display(precios.head())

**Interpretación:** cada columna representa un activo y cada fila una fecha común. Una diferencia en niveles de precio no significa que un activo sea más rentable: para comparar variaciones se utilizan rendimientos y otros indicadores apropiados.

### Exportar para el siguiente módulo

El archivo CSV conservará los precios preparados. Guardaremos también los rendimientos como un archivo independiente. Es importante distinguir los datos originales de los transformados.

In [ ]:
rendimientos = precios.pct_change(fill_method=None).dropna(how="any")
precios.to_csv("precios_portafolio.csv", index_label="Fecha")
rendimientos.to_csv("rendimientos_portafolio.csv", index_label="Fecha")
print("Archivos creados: precios_portafolio.csv y rendimientos_portafolio.csv")
display(rendimientos.head())

## 8 · Reto integrador: prepara tu base financiera

Elabora un procedimiento propio con **tres activos** de tu elección (pueden ser acciones o ETF disponibles en la fuente). Documenta:

1. La pregunta financiera y la razón para seleccionar esos activos.
2. Los tickers, el período consultado y las opciones de descarga.
3. Las dimensiones, columnas, tipos de datos e índice de cada consulta.
4. El diagnóstico de valores faltantes, fechas duplicadas y precios no positivos.
5. Las transformaciones realizadas y su justificación.
6. Una tabla de precios alineados por fechas y otra de rendimientos.
7. Una comprobación manual de al menos un rendimiento.
8. Dos gráficos y una interpretación escrita de **150 a 200 palabras**, incluyendo limitaciones de los datos.

**Criterio de calidad:** no elimines ni reemplaces datos sin explicar por qué. Conserva las evidencias de la validación y reconoce las limitaciones de la fuente.

**Producto:** notebook ejecutable y dos archivos CSV generados por el procedimiento.

In [ ]:
# RETO INTEGRADOR — SOLUCIÓN ORIGINAL
# Consulta tus activos, verifica los datos, transforma,
# comprueba rendimientos y exporta los resultados.

## 9 · Autoevaluación y continuidad

- [ ] Distingo la fuente de información del instrumento financiero.
- [ ] Explico qué es un DataFrame, su índice y sus columnas.
- [ ] Identifico fechas, tipos y observaciones faltantes.
- [ ] Justifico decisiones de limpieza y alineación temporal.
- [ ] Calculo rendimientos y verifico la fórmula.
- [ ] Distingo el significado de un gráfico de precios y uno de rendimientos.
- [ ] Documento parámetros, supuestos y limitaciones.
- [ ] Exporto datos preparados de forma reproducible.

**Próximo módulo:** utilizaremos las series preparadas para estudiar estadística descriptiva, dispersión, distribuciones y visualización financiera.

**Referencias:** [pandas](https://pandas.pydata.org/docs/) · [yfinance](https://ranaroussi.github.io/yfinance/) · [Python](https://docs.python.org/es/3/tutorial/).

**Advertencia:** el acceso a datos depende de servicios externos. Si una consulta falla, verifica conectividad y disponibilidad; no inventes precios para completar el ejercicio.

## Quiz de cierre · Tratamiento de datos financieros

**Quiz evaluativo 2 de 4 · componente de quices (15 % total)**

Responde individualmente las cinco preguntas. Las respuestas deben ser originales, razonadas y coherentes con el código que ejecutes. No basta con escribir una definición: explica el significado de los resultados cuando se solicite.

**Criterio sugerido:** cada pregunta vale 1 punto (5 puntos en total). Las preguntas 1–2 evalúan comprensión, 3–4 interpretación y 5 aplicación. La docente verificará las respuestas y registrará la nota conforme a la ponderación institucional de cada quiz.

### Pregunta 1 · Comprensión

¿Qué representa una fila y qué representa una columna en un DataFrame de precios históricos?

**Tu respuesta:**

_Escribe aquí tu explicación. Si se solicita código, utiliza la celda de Python siguiente._

### Pregunta 2 · Comprensión

¿Por qué debemos revisar valores faltantes y tipos de datos antes de calcular rendimientos?

**Tu respuesta:**

_Escribe aquí tu explicación. Si se solicita código, utiliza la celda de Python siguiente._

### Pregunta 3 · Interpretación

Explica la diferencia entre `df.head()` y `df.info()`.

**Tu respuesta:**

_Escribe aquí tu explicación. Si se solicita código, utiliza la celda de Python siguiente._

### Pregunta 4 · Interpretación

¿Qué significa el primer valor faltante que suele aparecer al aplicar `pct_change()` a una serie de precios?

**Tu respuesta:**

_Escribe aquí tu explicación. Si se solicita código, utiliza la celda de Python siguiente._

### Pregunta 5 · Aplicación

Con una serie de precios `pd.Series([100, 105, 102])`, calcula los rendimientos con pandas y explica uno de ellos.

**Tu respuesta:**

_Escribe aquí tu explicación. Si se solicita código, utiliza la celda de Python siguiente._



In [ ]:
# Pregunta 5 — solución individual
# Escribe y ejecuta aquí tu código.
